# Exact rational certificate for Lemma 4: exclusion of the extraneous branch

This notebook verifies, using exact rational arithmetic, the branch-selection step used in Lemma 4 of the paper.

The goal is to prove that the candidate $s_0^\ast$ in the rectangle
$$
I=\{x+iy:\ x\in[0.000160734,0.000160735],\
y\in[-0.006419166,-0.006419165]\}
$$
satisfies
$$
f_4(s_0^\ast)-f_5(s_0^\ast)\sqrt{f_3(s_0^\ast)}\ne 0.
$$

The notebook assumes the external certificate, obtained for example by `RealRootCounting`, that the squared equation has a unique root $s_0^\ast$ in $I$. Everything after that point is verified exactly with rational arithmetic.

No floating-point arithmetic is used in the verification cells.


## Proof strategy

Let
$$
\widehat{s}_0=\frac{160734}{10^9}-\frac{6419165}{10^9}i
$$
be the upper-left corner of $I$. We prove exact perturbation bounds from $\widehat{s}_0$ to any $s\in I$, hence in particular to $s_0^\ast$.

It is enough to prove:
$$
f_5(s_0^\ast)\ne 0,\qquad
\operatorname{Im} f_3(s_0^\ast)>0,\qquad
\operatorname{Im}\frac{f_4(s_0^\ast)}{f_5(s_0^\ast)}<0.
$$
Then $\sqrt{f_3(s_0^\ast)}$ lies in the upper half-plane under the principal-square-root convention
$$
\sqrt{re^{i\theta}}=\sqrt r\,e^{i\theta/2},\qquad \theta\in(-\pi,\pi],
$$
whereas $f_4(s_0^\ast)/f_5(s_0^\ast)$ lies in the lower half-plane. Therefore they are not equal, and hence
$$
f_4(s_0^\ast)-f_5(s_0^\ast)\sqrt{f_3(s_0^\ast)}\ne0.
$$


In [1]:
from fractions import Fraction
from dataclasses import dataclass

@dataclass(frozen=True)
class CQ:
    """A complex number with rational real and imaginary parts."""
    re: Fraction
    im: Fraction

    def __add__(self, other):
        return CQ(self.re + other.re, self.im + other.im)

    def __sub__(self, other):
        return CQ(self.re - other.re, self.im - other.im)

    def __mul__(self, other):
        return CQ(self.re * other.re - self.im * other.im,
                  self.re * other.im + self.im * other.re)

    def conj(self):
        return CQ(self.re, -self.im)

    def norm2(self):
        return self.re * self.re + self.im * self.im

    def inv(self):
        n2 = self.norm2()
        return CQ(self.re / n2, -self.im / n2)

    def __truediv__(self, other):
        return self * other.inv()

def cq_pow(z: CQ, n: int) -> CQ:
    out = CQ(Fraction(1), Fraction(0))
    for _ in range(n):
        out = out * z
    return out

def poly_eval(coeffs: dict[int, int], z: CQ) -> CQ:
    """Evaluate sum_k coeffs[k] z^k exactly."""
    out = CQ(Fraction(0), Fraction(0))
    for k, a in coeffs.items():
        if a != 0:
            zk = cq_pow(z, k)
            out = out + CQ(Fraction(a) * zk.re, Fraction(a) * zk.im)
    return out

def lipschitz_bound(coeffs: dict[int, int], R: Fraction, delta: Fraction) -> Fraction:
    r"""Return delta * sum_{k>=1} |a_k| k R^{k-1}.

    If |z|,|w| <= R and |z-w| <= delta, then
        |p(z)-p(w)| <= this bound.
    """
    total = Fraction(0)
    for k, a in coeffs.items():
        if k >= 1 and a != 0:
            total += abs(a) * k * (R ** (k - 1))
    return delta * total

def im_div_numerator(A: CQ, B: CQ) -> Fraction:
    r"""Numerator of Im(A/B):
        Im(A/B) = (A.im * B.re - A.re * B.im)/|B|^2.
    """
    return A.im * B.re - A.re * B.im

def frac_to_latex(q: Fraction) -> str:
    if q.denominator == 1:
        return str(q.numerator)
    return rf"\frac{{{q.numerator}}}{{{q.denominator}}}"

print("Exact rational arithmetic initialized.")


Exact rational arithmetic initialized.


## Input polynomials

The relevant polynomials are
$$
\begin{aligned}
f_3(z)&=4z^4-108z^3-1439z^2-108z+4,\\
f_4(z)&=-8192z^{16}+499712z^{15}-7077888z^{14}-31111168z^{13}
+659681792z^{12}\\
&\quad -326163456z^{11}-5979808768z^{10}+7396888576z^9
+3606093312z^8\\
&\quad -22677711616z^7+2524591872z^6+1298253312z^5
-54335488z^4\\
&\quad -8839168z^3+552960z^2-8192z,\\
f_5(z)&=-4096z^{14}+194560z^{13}-2022400z^{12}-5172736z^{11}
+72910848z^{10}\\
&\quad -65459200z^9-205725696z^8+369364992z^7-284532480z^6\\
&\quad -194358528z^5+16606208z^4+2543616z^3-221184z^2+4096z.
\end{aligned}
$$


In [2]:
f3 = {
    4: 4,
    3: -108,
    2: -1439,
    1: -108,
    0: 4,
}

f4 = {
    16: -8192,
    15: 499712,
    14: -7077888,
    13: -31111168,
    12: 659681792,
    11: -326163456,
    10: -5979808768,
    9: 7396888576,
    8: 3606093312,
    7: -22677711616,
    6: 2524591872,
    5: 1298253312,
    4: -54335488,
    3: -8839168,
    2: 552960,
    1: -8192,
}

f5 = {
    14: -4096,
    13: 194560,
    12: -2022400,
    11: -5172736,
    10: 72910848,
    9: -65459200,
    8: -205725696,
    7: 369364992,
    6: -284532480,
    5: -194358528,
    4: 16606208,
    3: 2543616,
    2: -221184,
    1: 4096,
}

print("Polynomials loaded with integer coefficients.")


Polynomials loaded with integer coefficients.


## Exact rational rectangle and overestimates

We use rational endpoints:
$$
x\in\left[\frac{160734}{10^9},\frac{160735}{10^9}\right],\qquad
y\in\left[-\frac{6419166}{10^9},-\frac{6419165}{10^9}\right].
$$

The approximation point is
$$
\widehat{s}_0=\frac{160734}{10^9}-\frac{6419165}{10^9}i.
$$

For every $s\in I$,
$$
|s-\widehat{s}_0|\le \sqrt{2}\,10^{-9}<2\cdot 10^{-9}=:\delta.
$$
We also use the rational radius
$$
R=\frac1{100}
$$
so that $|s|\le R$ for all $s\in I$ and $|\widehat{s}_0|\le R$.


In [3]:
# Rectangle endpoints
xL = Fraction(160734, 10**9)
xU = Fraction(160735, 10**9)
yL = Fraction(-6419166, 10**9)
yU = Fraction(-6419165, 10**9)

s_hat = CQ(xL, yU)

# Exact rational overestimates
delta = Fraction(1, 500_000_000)  # 2e-9
R = Fraction(1, 100)

# Verify |s - s_hat| <= delta for all s in I by checking the farthest corner.
dx = xU - xL
dy = yU - yL
assert dx == Fraction(1, 10**9)
assert dy == Fraction(1, 10**9)
assert dx*dx + dy*dy < delta*delta

# Verify the entire rectangle lies inside |z| <= R.
# Since x is positive and y is negative, the largest modulus occurs at (xU, yL).
max_norm2 = xU*xU + yL*yL
assert max_norm2 < R*R
assert s_hat.norm2() < R*R

print("Exact rectangle checks passed.")
print("delta =", delta)
print("R =", R)


Exact rectangle checks passed.
delta = 1/500000000
R = 1/100


## Perturbation bound for polynomials

For $p(z)=\sum_k a_kz^k$, if $|z|,|w|\le R$ and $|z-w|\le \delta$, then
$$
|p(z)-p(w)|
\le
\delta\sum_{k\ge1}|a_k|\,k R^{k-1}.
$$
This follows from
$$
z^k-w^k=(z-w)\sum_{j=0}^{k-1}z^{k-1-j}w^j.
$$


In [4]:
# Exact values at s_hat
F3 = poly_eval(f3, s_hat)
F4 = poly_eval(f4, s_hat)
F5 = poly_eval(f5, s_hat)

# Exact rational perturbation bounds
E3 = lipschitz_bound(f3, R, delta)
E4 = lipschitz_bound(f4, R, delta)
E5 = lipschitz_bound(f5, R, delta)

print("Computed exact values and exact perturbation bounds.")
print("F3.re =", F3.re)
print("F3.im =", F3.im)
print("E3    =", E3)
print("F4.re =", F4.re)
print("F4.im =", F4.im)
print("E4    =", E4)
print("F5.re =", F5.re)
print("F5.im =", F5.im)
print("E5    =", E5)


Computed exact values and exact perturbation bounds.
F3.re = 1010475168890125634304392523786775561/250000000000000000000000000000000000
F3.im = 4351317281473980640131720181605459/6250000000000000000000000000000000
E3    = 1068847/3906250000000
F4.re = -2929963390422073850344825787069283634134521898118982629004316752663475476441350842285054812602456078112879754398184995995289144069707022558241/122070312500000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
F4.im = 37451009914612932873937825035689989346624581107711593519775357709673157757787873049970170958933688120277385682235399862181196908251087637621/762939453125000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
E4    = 677089807492340436892204200379/15258789062500000000000000000000000
F5.re = 2378907202095121952366632423763725110581071403596951488973181584082821831594708163603275415

## Certificate 1: $f_5(s_0^\ast)\ne0$

Since $|f_5(s_0^\ast)-f_5(\widehat{s}_0)|\le E_5$, it is enough to verify
$$
E_5<|f_5(\widehat{s}_0)|.
$$
To avoid square roots, the notebook checks the equivalent rational inequality
$$
E_5^2<|f_5(\widehat{s}_0)|^2.
$$


In [5]:
assert E5 * E5 < F5.norm2()
print("Certificate 1 passed: E5^2 < |F5|^2, hence f5(s*) != 0.")


Certificate 1 passed: E5^2 < |F5|^2, hence f5(s*) != 0.


## Certificate 2: $f_3(s_0^\ast)$ lies in the upper half-plane

Since
$$
|\operatorname{Im} f_3(s_0^\ast)-\operatorname{Im} f_3(\widehat{s}_0)|
\le E_3,
$$
it is enough to verify
$$
\operatorname{Im} f_3(\widehat{s}_0)-E_3>0.
$$


In [6]:
assert F3.im - E3 > 0
print("Certificate 2 passed: Im f3(s*) > 0.")


Certificate 2 passed: Im f3(s*) > 0.


## Certificate 3: $f_4(s_0^\ast)/f_5(s_0^\ast)$ lies in the lower half-plane

Write
$$
F_4=f_4(\widehat{s}_0)=a+ib,\qquad F_5=f_5(\widehat{s}_0)=c+id.
$$
Then
$$
\operatorname{Im}\frac{F_4}{F_5}
=
\frac{bc-ad}{|F_5|^2}.
$$
For the exact values, define
$$
N_0=bc-ad.
$$

For the actual values at $s_0^\ast$, let the errors in $f_4$ and $f_5$ be bounded by $E_4$ and $E_5$. Since each real and imaginary component error is bounded by the corresponding complex modulus error, the numerator perturbation is bounded by
$$
E_N=(|a|+|b|)E_5+(|c|+|d|)E_4+2E_4E_5.
$$
Thus if
$$
N_0+E_N<0,
$$
then the actual numerator is strictly negative. Together with $f_5(s_0^\ast)\ne0$, this proves
$$
\operatorname{Im}\frac{f_4(s_0^\ast)}{f_5(s_0^\ast)}<0.
$$


In [7]:
a, b = F4.re, F4.im
c, d = F5.re, F5.im

N0 = im_div_numerator(F4, F5)
EN = (abs(a) + abs(b)) * E5 + (abs(c) + abs(d)) * E4 + 2 * E4 * E5

assert N0 + EN < 0
print("Certificate 3 passed: Im(f4(s*)/f5(s*)) < 0.")
print("N0 =", N0)
print("EN =", EN)


Certificate 3 passed: Im(f4(s*)/f5(s*)) < 0.
N0 = -187136899121820860955756449222310955879608341775568673236852273536627796423966354454419297274631696529952071490743881115401619304335338141653308116234762891263336584969609199330887903109115845104848351119841402898903264904533085245359188877107154259378501845360031/1490116119384765625000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
EN = 1086643016899215955603041793563189468042286906281418345128698770386966226725874138423397479588203423901661406432841063092856205194650702583679860787707643091822410455791/372529029846191406250000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


## Conclusion

The exact rational assertions above prove:
$$
f_5(s_0^\ast)\ne0,\qquad
\operatorname{Im} f_3(s_0^\ast)>0,\qquad
\operatorname{Im}\frac{f_4(s_0^\ast)}{f_5(s_0^\ast)}<0.
$$

Therefore $\sqrt{f_3(s_0^\ast)}$ lies in the upper half-plane under the stated principal-square-root convention, whereas $f_4(s_0^\ast)/f_5(s_0^\ast)$ lies in the lower half-plane. Hence
$$
\frac{f_4(s_0^\ast)}{f_5(s_0^\ast)}
\ne
\sqrt{f_3(s_0^\ast)}.
$$
Since $f_5(s_0^\ast)\ne0$, this gives
$$
f_4(s_0^\ast)-f_5(s_0^\ast)\sqrt{f_3(s_0^\ast)}\ne0.
$$

This is the required exclusion of the extraneous branch introduced by squaring.


## Optional: LaTeX-ready certificate lines

The following cell prints compact exact inequalities that can be pasted into an appendix. The proof above does not depend on decimal approximations.


In [8]:
print("LaTeX certificate:")
print(r"$$")
print(r"E_5^2 < |f_5(\widehat{s}_0)|^2")
print(r"$$")
print("Left  =", frac_to_latex(E5 * E5))
print("Right =", frac_to_latex(F5.norm2()))

print(r"$$")
print(r"\operatorname{Im} f_3(\widehat{s}_0)-E_3>0")
print(r"$$")
print("Value =", frac_to_latex(F3.im - E3))

print(r"$$")
print(r"N_0+E_N<0")
print(r"$$")
print("Value =", frac_to_latex(N0 + EN))


LaTeX certificate:
$$
E_5^2 < |f_5(\widehat{s}_0)|^2
$$
Left  = \frac{3263070990057471281565624617781886356558578570424481}{9313225746154785156250000000000000000000000000000000000000000}
Right = \frac{43390487750827218434278488316083247619321834331504179062843226840817263411634661897286660012859789791619829404389470705138485504092118482119754216788733289166404059696033525393776682003422166508040043249777800862340023567937675420272764259515468121}{59604644775390625000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000}
$$
\operatorname{Im} f_3(\widehat{s}_0)-E_3>0
$$
Value = \frac{4351315571318780640131720181605459}{6250000000000000000000000000000000}
$$
N_0+E_N<0
$$
Value = \frac{-1871325525497532640919340370551367031217361726279435475634717587415462485590594509578656036847133437162564648451181497511492478795145595